In [ ]:
import pandas as pd
import requests
import pandas_datareader.data as web
import datetime

# ----------------------------------------------------
# 1. EXTRACCIÓN del BC
# ----------------------------------------------------
USER_BCCH = "xxxx"
PASS_BCCH = "xxx"

# Diccionario con el código de la serie y su nombre
series_bcch = {
    "F073.TCO.PRE.HIST.M": "Tipo_Cambio",
    "F022.TPM.TIN.D001.NO.Z.M": "TPM",
    "F074.IPC.IND.Z.EP09.C.M": "IPC",
    "F032.IMC.IND.Z.Z.EP18.Z.Z.0.M": "IMACEC",
    "F019.PPB.PRE.40.M": "Precio_cobre",
    "F073.TCR.IND.199101.M": "TCR",
    "F051.H121.STO.H.C.CLP.M": "Reservas_internacionales",
    "F049.DES.TAS.INE9.10.M": "Desempleo"
}

def get_bcch_data(serie, user, password, start_date="2010-01-01"):
    url = f"https://si3.bcentral.cl/SieteRestWS/SieteRestWS.ashx?user={user}&pass={password}&firstdate={start_date}&timeseries={serie}&function=GetSeries"
    response = requests.get(url)
    if response.status_code == 200:
        # El BCCh devuelve un JSON, extraemos las fechas y valores
        data = response.json()['Series']['Obs']
        df = pd.DataFrame(data)
        df['indexDateString'] = pd.to_datetime(df['indexDateString'], format='%d-%m-%Y')
        df.set_index('indexDateString', inplace=True)
        df.rename(columns={'value': serie}, inplace=True)
        # Convertir a numérico, forzando NaN en caso de errores
        df[serie] = pd.to_numeric(df[serie], errors='coerce') 
        return df[[serie]]
    else:
        print(f"Error al descargar {serie}")
        return pd.DataFrame()

# Descargar y unir datos del BCCh
df_chile = pd.DataFrame()
for codigo, nombre in series_bcch.items():
    df_temp = get_bcch_data(codigo, USER_BCCH, PASS_BCCH, "2013-01-01")
    df_temp.rename(columns={codigo: nombre}, inplace=True)
    if df_chile.empty:
        df_chile = df_temp
    else:
        df_chile = df_chile.join(df_temp, how='outer')

# ----------------------------------------------------
# 2. EXTRACCIÓN DESDE FRED (EE.UU.)
# ----------------------------------------------------
start = datetime.datetime(2013, 1, 1)
end = datetime.datetime.now()

series_fred = {
    'FEDFUNDS': 'Tasa_FED_EEUU', # Tasa de interés de EE.UU.
    'CPIAUCSL': 'IPC_EEUU',      # Inflación de EE.UU.
    'DTWEXBGS': 'Indice_Dolar',  # Valor del dólar (Equivalente al DXY en FRED)
    'VIXCLS': 'Indice_VIX'       # Volatilidad del mercado / Riesgo global
}

df_usa = pd.DataFrame()
for codigo, nombre in series_fred.items():
    df_temp = web.DataReader(codigo, 'fred', start, end)
    df_temp.rename(columns={codigo: nombre}, inplace=True)
    if df_usa.empty:
        df_usa = df_temp
    else:
        df_usa = df_usa.join(df_temp, how='outer')

# ----------------------------------------------------
# 3. CONSOLIDACIÓN DE DATOS
# ----------------------------------------------------
# Unimos Chile y USA
df_final = df_chile.join(df_usa, how='outer')

# Como puede haber datos diarios y mensuales, remuestreamos todo a mensual (promedio)
df_mensual = df_final.resample('ME').mean()

print(df_mensual.tail())

                 Tipo_Cambio  TPM  IPC      IMACEC  Precio_cobre         TCR  \
indexDateString                                                                
2026-04-30        897.886190  4.5  4.0  114.361518      5.847439  101.762383   
2026-05-31        897.637895  4.5  3.9  112.106264      6.126738  102.235396   
2026-06-30        903.384286  4.5  4.3  110.601447      6.157215  102.211559   
2026-07-31        930.970000  4.5  3.5         NaN      6.134921  105.025856   
2026-08-31               NaN  NaN  NaN         NaN           NaN         NaN   

                 Reservas_internacionales  Desempleo  Tasa_FED_EEUU  IPC_EEUU  \
indexDateString                                                                 
2026-04-30                   46480.853071   9.108729           3.64   332.407   
2026-05-31                   46654.100662   9.437821           3.63   333.979   
2026-06-30                   47847.625303   9.440585           3.63   332.568   
2026-07-31                   48727